<div align="center">
<h1>🔐 KRİPTOQRAFİYA KURSU</h1>
<h2>Məşğələ 7</h2>
<h2>Müasir axın şifrələri</h2>
<h3 style="color: #8B4513;">eSTREAM, ChaCha20, Trivium və ARX dizaynı</h3>
<br>
<h3>Məşğələ vaxtı: 2 saat</h3>
<br>
<p><em>Hazırlanma tarixi: 2024</em></p>
</div>

## 📑 Mündəricat

1. [Məşğələnin məqsədləri](#məşğələnin-məqsədləri)
2. [Lazım olan kitabxanalar](#lazım-olan-kitabxanalar)
3. [Hazırlıq (15 dəq)](#hazırlıq-15-dəq)
4. [Xatırlatma: eSTREAM layihəsi (10 dəq)](#xatırlatma-estream-layihəsi-10-dəq)
5. [Trivium şifrəsinin implementasiyası (30 dəq)](#trivium-şifrəsinin-implementasiyası-30-dəq)
6. [Salsa20 və ChaCha20 (35 dəq)](#salsa20-və-chacha20-35-dəq)
7. [ChaCha20-Poly1305 AEAD (25 dəq)](#chacha20-poly1305-aead-25-dəq)
8. [eSTREAM portfelinin digər şifrələri (20 dəq)](#estream-portfelinin-digər-şifrələri-20-dəq)
9. [İnteqrasiya edilmiş tətbiq (15 dəq)](#inteqrasiya-edilmiş-tətbiq-15-dəq)
10. [Ev tapşırığı](#ev-tapşırığı)
11. [Yekun və müzakirə sualları](#yekun-və-müzakirə-sualları)
12. [Əlavə resurslar](#əlavə-resurslar)

## 🎯 Məşğələnin məqsədləri

Bu məşğələni bitirdikdən sonra siz:

✅ eSTREAM layihəsi və onun portfelinə daxil olan şifrələri izah edə biləcəksiniz  
✅ Trivium şifrəsinin quruluşunu başa düşəcək və implementasiya edə biləcəksiniz  
✅ Salsa20 və ChaCha20-nin ARX dizaynını izah edə biləcəksiniz  
✅ ChaCha20-Poly1305 AEAD rejimini istifadə edə biləcəksiniz  
✅ Müasir axın şifrələrini LFSR əsaslı şifrələrlə müqayisə edə biləcəksiniz  
✅ TLS 1.3 və mobil cihazlarda ChaCha20-nin rolunu başa düşəcəksiniz

## 📚 Lazım olan kitabxanalar

| Kitabxana | Quraşdırma | İstifadə sahəsi |
|-----------|------------|-----------------|
| Python standart kitabxanası | `hashlib`, `struct` (daxili) | Hashing, byte çevirmələri |
| `cryptography` | `!pip install cryptography` | Müasir kriptoqrafik əməliyyatlar |
| `matplotlib` | `!pip install matplotlib` | Qrafiklər çəkmək üçün (isteğə bağlı) |
| `numpy` | `!pip install numpy` | Rəqəmsal hesablamalar (isteğə bağlı) |

> 💡 **Qeyd:** `cryptography` kitabxanası əsas kitabxanadır. Digərləri isteğə bağlıdır.

In [ ]:
# Lazımi kitabxanaların yoxlanılması

import importlib.util

required = ["cryptography", "matplotlib", "numpy"]
missing = [pkg for pkg in required if importlib.util.find_spec(pkg) is None]

if missing:
    print("⚠️ Aşağıdakı kitabxanalar bu mühitdə tapılmadı:")
    for pkg in missing:
        print(f"   - {pkg}")
    print("\nQuraşdırmaq üçün:")
    print("pip install " + " ".join(missing))
else:
    print("✅ Lazımi kitabxanalar artıq mövcuddur")

## 🔧 Hazırlıq (15 dəq)

### 3.1 Python mühitinin yoxlanılması

Aşağıdakı kodu işə salaraq lazımi modulların yükləndiyini yoxlayın:

In [ ]:
import sys
import os
import hashlib
import struct
from collections import Counter

# Kriptoqrafiya kitabxanası
try:
    from cryptography.hazmat.primitives.ciphers import Cipher, algorithms, modes
    from cryptography.hazmat.backends import default_backend
    print("✅ cryptography kitabxanası yüklü")
    HAS_CRYPTOGRAPHY = True
except ImportError:
    print("❌ cryptography kitabxanası yoxdur")
    print("   Quraşdırmaq üçün: pip install cryptography")
    HAS_CRYPTOGRAPHY = False

# Əlavə kitabxanalar
try:
    import numpy as np
    import matplotlib.pyplot as plt
    print("✅ numpy və matplotlib yüklü - qrafiklər çəkilə bilər")
    HAS_NUMPY = True
    HAS_MATPLOTLIB = True
except ImportError:
    print("⚠️ numpy və ya matplotlib yoxdur (istəyə bağlı)")
    np = None
    plt = None
    HAS_NUMPY = False
    HAS_MATPLOTLIB = False

print(f"\n🐍 Python versiyası: {sys.version}")
if HAS_CRYPTOGRAPHY:
    print("📦 Bütün lazımi modullar yükləndi!")
else:
    print("⛔ Lazımi modullar tam yüklənmədi. Davam etməzdən əvvəl cryptography kitabxanasını quraşdırın və bu hüceyrəni yenidən işə salın.")

### 3.2 İşçi qovluğun yaradılması

In [ ]:
import os
from pathlib import Path

# İşçi qovluğu yaradaq
# İlk işə salınan qovluğu yadda saxlayırıq ki, bu hüceyrə təkrar işə salınanda
# crypto_workshop/lecture7/crypto_workshop/lecture7 kimi iç-içə qovluqlar yaratmasın.
if "_lecture7_start_dir" not in globals():
    _lecture7_start_dir = Path.cwd().resolve()

workspace = _lecture7_start_dir / "crypto_workshop" / "lecture7"
workspace.mkdir(parents=True, exist_ok=True)
os.chdir(workspace)

print(f"📁 İşçi qovluq: {os.getcwd()}")
print(f"📂 Qovluğun məzmunu: {os.listdir('.')}")

## 📖 Xatırlatma: eSTREAM layihəsi (10 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 eSTREAM layihəsi</h4>
<p>eSTREAM (ECRYPT Stream Cipher Project) — 2004-2008-ci illərdə Avropa İttifaqının ECRYPT şəbəkəsi tərəfindən təşkil edilmiş layihədir. Məqsəd geniş istifadəyə yararlı yeni axın şifrələrini müəyyən etmək idi.</p>

<p><b>Profillər:</b></p>
<ul>
    <li><b>Profil 1 (proqram)</b>: Yüksək ötürmə sürəti tələb edən proqram təminatı üçün</li>
    <li><b>Profil 2 (aparat)</b>: Məhdud resurslu aparat tətbiqləri üçün yüngül şifrələr</li>
</ul>

<p><b>Final portfel:</b></p>
<ul>
    <li><b>Profil 1:</b> HC-128, Rabbit, Salsa20/12, SOSEMANUK</li>
    <li><b>Profil 2:</b> Grain, MICKEY, Trivium</li>
</ul>
</div>

Bu məşğələdə eSTREAM portfelinin ən əhəmiyyətli nümayəndələrini öyrənəcəyik:
* **Trivium** - ən yüngül aparat şifrəsi
* **Salsa20/ChaCha20** - ən populyar proqram şifrəsi (TLS 1.3-də standart)

## 🔧 Trivium şifrəsinin implementasiyası (30 dəq)

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 Trivium</h4>
<p>Trivium, <b>Christophe De Cannière və Bart Preneel</b> tərəfindən hazırlanmış, eSTREAM layihəsinin Profil 2 (aparat) portfelinə seçilmiş yüngül axın şifrəsidir.</p>
<ul>
    <li><b>Açar:</b> 80-bit</li>
    <li><b>IV:</b> 80-bit</li>
    <li><b>Daxili vəziyyət:</b> 288-bit (3 registr: 93, 84, 111)</li>
    <li><b>İlkinləşdirmə:</b> 1152 dövr</li>
    <li><b>Aparat mürəkkəbliyi:</b> 3488 qapı - RFID və sensor şəbəkələri üçün ideal</li>
</ul>
</div>

In [ ]:
class Trivium:
    """
    Trivium axın şifrəsinin implementasiyası

    eSTREAM Profil 2 (aparat) portfelinin ən yüngül şifrəsi
    """

    def __init__(self, key, iv):
        """
        key: 80-bit açar (bytes və ya bit siyahısı)
        iv: 80-bit ilkin vektor (bytes və ya bit siyahısı)
        """
        # Açar və IV-i bit siyahısına çevir
        if isinstance(key, bytes):
            self.key = self.bytes_to_bits(key, 80)
        else:
            self.key = list(key)[:80]

        if isinstance(iv, bytes):
            self.iv = self.bytes_to_bits(iv, 80)
        else:
            self.iv = list(iv)[:80]

        # Daxili vəziyyəti ilkinləşdir
        self.state = self._init_state()
        self.initial_state = self.state.copy()
        self._warmed_up = False

    def bytes_to_bits(self, data, length):
        """Baytları bit siyahısına çevirir"""
        bits = []
        for byte in data:
            for i in range(7, -1, -1):
                bits.append((byte >> i) & 1)
        # Lazım gələrsə, sıfırlarla doldur
        while len(bits) < length:
            bits.insert(0, 0)
        return bits[:length]

    def bits_to_bytes(self, bits):
        """Bit siyahısını baytlara çevirir"""
        bytes_data = bytearray()
        for i in range(0, len(bits), 8):
            byte = 0
            for j in range(8):
                if i + j < len(bits):
                    byte = (byte << 1) | bits[i + j]
            bytes_data.append(byte)
        return bytes(bytes_data)

    def _init_state(self):
        """
        Daxili vəziyyəti ilkinləşdirir
        """
        # 288-bit vəziyyət: A (93), B (84), C (111)
        state = [0] * 288

        # A registri (0-92): açarın 80 biti, qalanlar 0
        for i in range(80):
            state[i] = self.key[i]

        # B registri (93-176): IV-in 80 biti, qalanlar 0
        for i in range(80):
            state[93 + i] = self.iv[i]

        # C registri (177-287): son 3 bit 1, qalanlar 0
        state[285] = 1
        state[286] = 1
        state[287] = 1

        return state

    def _warm_up(self):
        """
        1152 dövr ilkinləşdirmə (çıxış vermədən)
        """
        for _ in range(1152):
            self._next_bit()
        self._warmed_up = True

    def _next_bit(self):
        """
        Növbəti biti hesablayır və vəziyyəti yeniləyir
        """
        a = self.state
        b = self.state
        c = self.state

        # Çıxış bitini hesabla
        t1 = a[65] ^ a[92]
        t2 = b[161] ^ b[176]
        t3 = c[242] ^ c[287]
        z = t1 ^ t2 ^ t3

        # Feedback bitlərini hesabla
        t1 = t1 ^ (a[90] & a[91]) ^ a[170]
        t2 = t2 ^ (b[174] & b[175]) ^ b[263]
        t3 = t3 ^ (c[285] & c[286]) ^ c[68]

        # Vəziyyəti yenilə (sola sürüşdür)
        self.state = [t3] + self.state[:92] + [t1] + self.state[93:176] + [t2] + self.state[177:287]

        return z

    def generate_keystream(self, n):
        """
        n bit açar axını generasiya edir
        """
        # İlkinləşdirmə (əgər hələ edilməyibsə)
        if not self._warmed_up:
            self._warm_up()

        bits = []
        for _ in range(n):
            bits.append(self._next_bit())

        return bits

    def encrypt(self, plaintext):
        """
        Mətni şifrləyir
        """
        if isinstance(plaintext, str):
            plaintext = plaintext.encode('utf-8')

        # Açar axını generasiya et (8 bit * bayt sayı)
        keystream_bits = self.generate_keystream(len(plaintext) * 8)

        ciphertext = bytearray()
        for i, byte in enumerate(plaintext):
            # Bu bayt üçün 8 bit açar götür
            key_byte = 0
            for j in range(8):
                key_byte = (key_byte << 1) | keystream_bits[i*8 + j]

            ciphertext.append(byte ^ key_byte)

        return bytes(ciphertext)

    def decrypt(self, ciphertext):
        """
        Şifrəli mətni deşifrləyir
        """
        return self.encrypt(ciphertext)

    def reset(self):
        """
        Şifrəni sıfırlayır
        """
        self.state = self.initial_state.copy()
        self._warmed_up = False

# Test
import secrets

print("=" * 70)
print("🔧 TRIVIUM TESTİ")
print("=" * 70)

# Test üçün açar və IV-i təsadüfi generasiya et (80 bit = 10 bayt)
key = secrets.token_bytes(10)
iv = secrets.token_bytes(10)

trivium = Trivium(key, iv)
plaintext = "Salam, bu Trivium şifrəsinin testidir!"
print(f"📨 Orijinal mətn: {plaintext}")

ciphertext = trivium.encrypt(plaintext)
print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")

trivium.reset()
decrypted = trivium.decrypt(ciphertext)
print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
print(f"\n✅ Uğurlu? {plaintext == decrypted.decode('utf-8')}")


### 5.1 Trivium-un analizi

Trivium-un ən mühüm xüsusiyyəti onun **yüngüllüyü**dür. Cəmi 3488 qapı ilə implementasiya oluna bilir ki, bu da onu RFID və sensor şəbəkələri üçün ideal edir.

In [ ]:
def analyze_trivium():
    """Trivium-un statistik xüsusiyyətlərini analiz edir"""
    print("\n" + "-" * 70)
    print("📊 TRIVIUM STATİSTİK ANALİZİ")
    print("-" * 70)

    # 1000 bit generasiya et
    trivium = Trivium(b'0123456789', b'abcdefghij')
    bits = trivium.generate_keystream(1000)

    # 1-lərin sayı
    ones = sum(bits)
    zeros = len(bits) - ones
    print(f"📈 1000 bitdə 1-lərin sayı: {ones} ({ones/10:.1f}%)")
    print(f"📉 0-ların sayı: {zeros} ({zeros/10:.1f}%)")
    print(f"   İdeal nisbət: 50%")

    # Monobit testi (sadələşdirilmiş)
    diff = abs(ones - 500)
    if diff < 30:
        print("✅ Monobit testi keçdi")
    else:
        print("⚠️ Monobit testi keçmədi")

    # Qrafik (əgər matplotlib varsa)
    if HAS_MATPLOTLIB:
        plt.figure(figsize=(12, 4))
        plt.stem(range(100), bits[:100], basefmt=' ')
        plt.title('Trivium - İlk 100 bit')
        plt.xlabel('Bit indeksi')
        plt.ylabel('Bit dəyəri')
        plt.ylim(-0.1, 1.1)
        plt.grid(True, alpha=0.3)
        plt.show()

analyze_trivium()

### ✍️ Çalışma 1: Trivium (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Period analizi (nəzəri):** Trivium sinfinə tam 288-bit vəziyyət üzərində exhaustive cycle detection əlavə etməyin — bu praktik olaraq mümkün deyil. Bunun əvəzinə 288-bit vəziyyət fəzası üçün periodun nəzəri yuxarı sərhədini hesablayın və qısa izah edin.

2. **Açar həssaslığı:** Fərqli açarlar və IV-lər üçün generasiya olunan açar axınlarını müqayisə edin.
   * Eyni açar, fərqli IV
   * Fərqli açar, eyni IV
   * 1 bit dəyişən açar

3. **Statistik testlər:** Trivium-un çıxışının təsadüfiliyini test edin:
   * Monobit testi (1-lərin nisbəti)
   * Runs testi (ardıcıl eyni bitlərin sayı)

4. **Nəzəri sual:** Niyə Trivium "yüngül" axın şifrəsi hesab olunur? Onun əsas üstünlükləri nələrdir?

5. **Təhlükəsizlik:** Trivium-un 80-bit təhlükəsizliyi nə deməkdir? Bu gün üçün kifayətdirmi?

In [ ]:
# Çalışma 1 - Cavablar

print("📝 ÇALIŞMA 1 CAVABLARI")
print("=" * 80)

# 1. Period analizi (nəzəri)
print("\n1. PERİOD ANALİZİ:")
print("   Trivium 288-bit vəziyyətə malikdir.")
print("   Maksimum mümkün period: 2^288 - 1 (nəzəri)")
print("   Lakin xətti mürəkkəblik və digər faktorlar səbəbindən praktik period daha kiçikdir.")

# 2. Açar həssaslığı
print("\n2. AÇAR HƏSSASLIĞI:")
trivium1 = Trivium(b'0123456789', b'abcdefghij')
trivium2 = Trivium(b'0123456789', b'abcdefghik')  # IV-də 1 bit dəyişik

bits1 = trivium1.generate_keystream(100)
bits2 = trivium2.generate_keystream(100)

diff_count = sum(1 for a, b in zip(bits1, bits2) if a != b)
print(f"   IV 1 bit dəyişdikdə fərqli bitlərin sayı: {diff_count}/100 ({diff_count}%)")
print(f"   İdeal avalanche effekti: təxminən 50%")

# 3. Runs testi
print("\n3. RUNS TESTİ:")
bits = trivium1.generate_keystream(1000)
runs = []
current_run = 1
for i in range(1, len(bits)):
    if bits[i] == bits[i-1]:
        current_run += 1
    else:
        runs.append(current_run)
        current_run = 1
runs.append(current_run)

avg_run = sum(runs) / len(runs)
print(f"   Orta run uzunluğu: {avg_run:.2f} bit")
print(f"   Maksimum run: {max(runs)} bit")

# 4. Yüngüllük
print("\n4. YÜNGÜLLÜK:")
print("   • Cəmi 3488 qapı ilə implementasiya oluna bilir")
print("   • RFID teqləri, sensor şəbəkələri üçün ideal")
print("   • Aşağı enerji sərfiyyatı")

# 5. Təhlükəsizlik
print("\n5. TƏHLÜKƏSİZLİK:")
print("   80-bit təhlükəsizlik = 2^80 əməliyyat tələb edir.")
print("   Bu gün üçün zəif hesab olunur (AES-128 tövsiyə olunur).")
print("   Lakin Trivium-un daha təhlükəsiz versiyaları (Trivium-128) mövcuddur.")

## 🔄 Salsa20 və ChaCha20 (35 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 ARX dizaynı</h4>
<p>ARX — <b>Add</b> (toplama), <b>Rotate</b> (döndərmə), <b>XOR</b> (exclusive or) əməliyyatlarına əsaslanan dizayn fəlsəfəsidir. S-qutuları istifadə etmədən qeyri-xəttilik təmin edir.</p>

<p><b>Üstünlükləri:</b></p>
<ul>
    <li>Bütün əməliyyatlar CPU-larda sürətli</li>
    <li>Sabit vaxt implementasiyası (timing attack-lara qarşı)</li>
    <li>Sadə və yığcam kod</li>
    <li>Heç bir S-qutusu yaddaşı tələb etmir</li>
</ul>
</div>

### 6.1 Salsa20 dörddəbir raundu və tam implementasiya

In [ ]:
import secrets


def _rotl32(x, n):
    """32-bit sola döndərmə"""
    x &= 0xffffffff
    return ((x << n) | (x >> (32 - n))) & 0xffffffff


def salsa20_quarter_round(a, b, c, d):
    """
    Salsa20 dörddəbir raund funksiyası (orijinal spesifikasiyaya uyğun).

    quarterround(y0, y1, y2, y3) = (z0, z1, z2, z3):
        z1 = y1 XOR ((y0 + y3) <<< 7)
        z2 = y2 XOR ((z1 + y0) <<< 9)
        z3 = y3 XOR ((z2 + z1) <<< 13)
        z0 = y0 XOR ((z3 + z2) <<< 18)
    """
    z1 = b ^ _rotl32((a + d) & 0xffffffff, 7)
    z2 = c ^ _rotl32((z1 + a) & 0xffffffff, 9)
    z3 = d ^ _rotl32((z2 + z1) & 0xffffffff, 13)
    z0 = a ^ _rotl32((z3 + z2) & 0xffffffff, 18)
    return z0, z1, z2, z3


def salsa20_row_round(y):
    """Salsa20 rowround funksiyası"""
    z = [0] * 16
    z[0], z[1], z[2], z[3] = salsa20_quarter_round(y[0], y[1], y[2], y[3])
    z[5], z[6], z[7], z[4] = salsa20_quarter_round(y[5], y[6], y[7], y[4])
    z[10], z[11], z[8], z[9] = salsa20_quarter_round(y[10], y[11], y[8], y[9])
    z[15], z[12], z[13], z[14] = salsa20_quarter_round(y[15], y[12], y[13], y[14])
    return z


def salsa20_column_round(x):
    """Salsa20 columnround funksiyası"""
    y = [0] * 16
    y[0], y[4], y[8], y[12] = salsa20_quarter_round(x[0], x[4], x[8], x[12])
    y[5], y[9], y[13], y[1] = salsa20_quarter_round(x[5], x[9], x[13], x[1])
    y[10], y[14], y[2], y[6] = salsa20_quarter_round(x[10], x[14], x[2], x[6])
    y[15], y[3], y[7], y[11] = salsa20_quarter_round(x[15], x[3], x[7], x[11])
    return y


def salsa20_double_round(x):
    """Bir cüt raund = columnround sonra rowround"""
    return salsa20_row_round(salsa20_column_round(x))


def salsa20_hash(input64):
    """64 baytlıq girişdən Salsa20 hash blokunu hesablayır"""
    if len(input64) != 64:
        raise ValueError("Salsa20 hash girişinin uzunluğu 64 bayt olmalıdır")

    words = [int.from_bytes(input64[i:i+4], 'little') for i in range(0, 64, 4)]
    working = words[:]

    for _ in range(10):
        working = salsa20_double_round(working)

    result = [(working[i] + words[i]) & 0xffffffff for i in range(16)]

    out = bytearray()
    for word in result:
        out.extend(word.to_bytes(4, 'little'))
    return bytes(out)


def salsa20_expand(key, nonce16):
    """
    Salsa20 expand funksiyası.

    key: 16 və ya 32 bayt
    nonce16: 16 bayt (8 bayt nonce + 8 bayt sayğac)
    """
    if len(nonce16) != 16:
        raise ValueError("nonce16 16 bayt olmalıdır")
    if len(key) not in (16, 32):
        raise ValueError("Açar 16 və ya 32 bayt olmalıdır")

    if len(key) == 32:
        constants = b"expand 32-byte k"
        k0, k1 = key[:16], key[16:]
    else:
        constants = b"expand 16-byte k"
        k0, k1 = key, key

    input64 = (
        constants[0:4] + k0 + constants[4:8] +
        nonce16 +
        constants[8:12] + k1 + constants[12:16]
    )
    return salsa20_hash(input64)


class Salsa20:
    """
    Salsa20 axın şifrəsi (DJB spesifikasiyasına uyğun).

    - 32 bayt açar (bu notebook üçün 256-bit variant)
    - 8 bayt nonce
    - 64-bit blok sayğacı
    """

    def __init__(self, key, nonce, counter=0):
        if len(key) != 32:
            raise ValueError("❌ Açar 32 bayt olmalıdır")
        if len(nonce) != 8:
            raise ValueError("❌ Nonce 8 bayt olmalıdır")

        self.key = key
        self.nonce = nonce
        self.counter = counter

    def _block(self, counter):
        counter_bytes = counter.to_bytes(8, 'little')
        return salsa20_expand(self.key, self.nonce + counter_bytes)

    def generate_keystream(self, length):
        keystream = bytearray()
        counter = self.counter

        while len(keystream) < length:
            keystream.extend(self._block(counter))
            counter += 1

        return bytes(keystream[:length])

    def encrypt(self, plaintext):
        if isinstance(plaintext, str):
            plaintext = plaintext.encode('utf-8')

        keystream = self.generate_keystream(len(plaintext))
        return bytes(p ^ k for p, k in zip(plaintext, keystream))

    def decrypt(self, ciphertext):
        return self.encrypt(ciphertext)


test_vectors = [
    (0x11111111, 0x22222222, 0x33333333, 0x44444444),
    (0x01234567, 0x89abcdef, 0x01234567, 0x89abcdef),
    (0x00000000, 0x00000000, 0x00000000, 0x00000000),
    (0xffffffff, 0xffffffff, 0xffffffff, 0xffffffff)
]

# Testlər
print("\n" + "=" * 70)
print("🔄 SALSA20 DÖRDƏBİR RAUND VƏ ŞİFRƏ TESTİ")
print("=" * 70)

# Rəsmi quarterround test vektoru
qr_in = (0x00000001, 0x00000000, 0x00000000, 0x00000000)
qr_expected = (0x08008145, 0x00000080, 0x00010200, 0x20500000)
qr_out = salsa20_quarter_round(*qr_in)
print(f"Quarterround nəticəsi: {[hex(x) for x in qr_out]}")
print(f"Rəsmi vektorla uyğun? {qr_out == qr_expected}")
assert qr_out == qr_expected, "Salsa20 quarterround spesifikasiya vektoruna uyğun deyil"

# Rəsmi expand test vektoru: bunlar açıq test dəyərləridir, real sirr deyil.
k0 = bytes(range(1, 17))
k1 = bytes(range(201, 217))
tv_key = k0 + k1
tv_nonce16 = bytes(range(101, 117))
expected_expand = bytes([
    69, 37, 68, 39, 41, 15,107,193,255,139,122,  6,170,233,217, 98,
    89,144,182,106, 21, 51,200, 65,239, 49,222, 34,215,114, 40,126,
   104,197,  7,225,197,153, 31,  2,102, 78, 76,176, 84,245,246,184,
   177,160,133,130,  6, 72,149,119,192,195,132,236,234,103,246, 74
])
expand_out = salsa20_expand(tv_key, tv_nonce16)
print(f"Expand vektoru uyğun? {expand_out == expected_expand}")
assert expand_out == expected_expand, "Salsa20 expand test vektoru uğursuz oldu"

# Sadə şifrləmə/deşifrləmə demo
# Real istifadədə açar təsadüfi, nonce isə eyni açarla təkrar istifadə olunmayan olmalıdır.
key = secrets.token_bytes(32)
nonce = secrets.token_bytes(8)
salsa = Salsa20(key, nonce)
plaintext = "Salsa20 artıq bu notebook-da düzgün implementasiya olunub."
ciphertext = salsa.encrypt(plaintext)
decrypted = Salsa20(key, nonce).decrypt(ciphertext)

print(f"📨 Orijinal mətn: {plaintext}")
print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")
print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
print(f"✅ Uğurlu? {plaintext == decrypted.decode('utf-8')}")


### 6.2 ChaCha20 dörddəbir raundu

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 ChaCha20</h4>
<p>ChaCha20, <b>Daniel Bernstein</b> tərəfindən 2008-ci ildə Salsa20-nin təkmilləşdirilmiş versiyası kimi hazırlanmışdır.</p>
<p><b>Əhəmiyyəti:</b></p>
<ul>
    <li>TLS 1.3-də müəyyən edilmiş və geniş dəstəklənən AEAD cipher suite-lərdən biridir; RFC 8446 üzrə mandatory-to-implement suite <code>TLS_AES_128_GCM_SHA256</code>-dir, <code>TLS_CHACHA20_POLY1305_SHA256</code> isə SHOULD implement-dir.</li>
    <li>Google tərəfindən mobil cihazlarda geniş istifadə olunur</li>
    <li>AES hardware sürətləndirilməsi olmayan cihazlarda AES-dən 3-5 dəfə sürətli</li>
</ul>
</div>

In [ ]:
def chacha20_quarter_round(a, b, c, d):
    """
    ChaCha20 dörddəbir raund funksiyası

    Salsa20-dən fərqli olaraq, toplama sonra XOR ardıcıllığı
    daha yaxşı diffuziya təmin edir
    """
    a = (a + b) & 0xffffffff
    d ^= a
    d = ((d << 16) | (d >> (32 - 16))) & 0xffffffff

    c = (c + d) & 0xffffffff
    b ^= c
    b = ((b << 12) | (b >> (32 - 12))) & 0xffffffff

    a = (a + b) & 0xffffffff
    d ^= a
    d = ((d << 8) | (d >> (32 - 8))) & 0xffffffff

    c = (c + d) & 0xffffffff
    b ^= c
    b = ((b << 7) | (b >> (32 - 7))) & 0xffffffff

    return a, b, c, d

# Test
print("\n" + "=" * 70)
print("🔄 CHACHA20 DÖRDƏBİR RAUND TESTİ")
print("=" * 70)

for i, (a, b, c, d) in enumerate(test_vectors):
    print(f"\nTest {i+1} - Giriş: {a:08x} {b:08x} {c:08x} {d:08x}")
    a, b, c, d = chacha20_quarter_round(a, b, c, d)
    print(f"        Çıxış: {a:08x} {b:08x} {c:08x} {d:08x}")

### 6.3 ChaCha20 blok funksiyası

In [ ]:
class ChaCha20:
    """
    ChaCha20 axın şifrəsi (RFC 8439)

    TLS 1.3-də standart şifrələmə alqoritmi
    """

    # Sabit sözlər ("expand 32-byte k" in ASCII)
    CONSTANTS = [0x61707865, 0x3320646e, 0x79622d32, 0x6b206574]

    def __init__(self, key, nonce, counter=0):
        """
        key: 32 bayt (256-bit) açar
        nonce: 12 bayt (96-bit) nonce
        counter: 4 bayt (32-bit) sayğac
        """
        if len(key) != 32:
            raise ValueError("❌ Açar 32 bayt olmalıdır")
        if len(nonce) != 12:
            raise ValueError("❌ Nonce 12 bayt olmalıdır")

        self.key = key
        self.nonce = nonce
        self.counter = counter
        self._state = None

    def _to_words(self, data):
        """Baytları 32-bit sözlərə çevirir"""
        return [int.from_bytes(data[i:i+4], 'little') for i in range(0, len(data), 4)]

    def _to_bytes(self, words):
        """32-bit sözləri baytlara çevirir"""
        result = bytearray()
        for w in words:
            result.extend(w.to_bytes(4, 'little'))
        return bytes(result)

    def _quarter_round(self, state, a, b, c, d):
        """Dörddəbir raund"""
        state[a] = (state[a] + state[b]) & 0xffffffff
        state[d] ^= state[a]
        state[d] = ((state[d] << 16) | (state[d] >> 16)) & 0xffffffff

        state[c] = (state[c] + state[d]) & 0xffffffff
        state[b] ^= state[c]
        state[b] = ((state[b] << 12) | (state[b] >> 20)) & 0xffffffff

        state[a] = (state[a] + state[b]) & 0xffffffff
        state[d] ^= state[a]
        state[d] = ((state[d] << 8) | (state[d] >> 24)) & 0xffffffff

        state[c] = (state[c] + state[d]) & 0xffffffff
        state[b] ^= state[c]
        state[b] = ((state[b] << 7) | (state[b] >> 25)) & 0xffffffff

        return state

    def _block(self, counter):
        """
        Bir blok (64 bayt) açar axını generasiya edir
        """
        # İlkin vəziyyət (4×4 matris)
        key_words = self._to_words(self.key)
        nonce_words = self._to_words(self.nonce)

        state = [
            self.CONSTANTS[0], self.CONSTANTS[1], self.CONSTANTS[2], self.CONSTANTS[3],
            key_words[0], key_words[1], key_words[2], key_words[3],
            key_words[4], key_words[5], key_words[6], key_words[7],
            counter, nonce_words[0], nonce_words[1], nonce_words[2]
        ]

        working = state.copy()

        # 20 raund (10 cüt raund)
        for _ in range(10):
            # Sütun raundları
            working = self._quarter_round(working, 0, 4, 8, 12)
            working = self._quarter_round(working, 1, 5, 9, 13)
            working = self._quarter_round(working, 2, 6, 10, 14)
            working = self._quarter_round(working, 3, 7, 11, 15)

            # Diaqonal raundlar
            working = self._quarter_round(working, 0, 5, 10, 15)
            working = self._quarter_round(working, 1, 6, 11, 12)
            working = self._quarter_round(working, 2, 7, 8, 13)
            working = self._quarter_round(working, 3, 4, 9, 14)

        # Son vəziyyət (ilkin + işlək)
        final = [(state[i] + working[i]) & 0xffffffff for i in range(16)]

        return self._to_bytes(final)

    def generate_keystream(self, length):
        """
        Verilmiş uzunluqda açar axını generasiya edir
        """
        keystream = bytearray()
        counter = self.counter

        while len(keystream) < length:
            block = self._block(counter)
            keystream.extend(block)
            counter += 1

        # Eyni obyektlə növbəti çağırışda açar axını təkrar istifadə olunmasın.
        self.counter = counter

        return bytes(keystream[:length])

    def encrypt(self, plaintext):
        """
        Mətni şifrləyir
        """
        if isinstance(plaintext, str):
            plaintext = plaintext.encode('utf-8')

        keystream = self.generate_keystream(len(plaintext))

        ciphertext = bytearray()
        for i, byte in enumerate(plaintext):
            ciphertext.append(byte ^ keystream[i])

        return bytes(ciphertext)

    def decrypt(self, ciphertext):
        """
        Şifrəli mətni deşifrləyir
        """
        return self.encrypt(ciphertext)

# Test
import secrets

print("\n" + "=" * 70)
print("🔐 CHACHA20 TESTİ")
print("=" * 70)

# 256-bit açar (32 bayt)
key = secrets.token_bytes(32)
# 96-bit nonce (12 bayt)
nonce = secrets.token_bytes(12)

chacha = ChaCha20(key, nonce)
plaintext = "Salam, bu ChaCha20 şifrəsinin testidir!"
print(f"📨 Orijinal mətn: {plaintext}")

ciphertext = chacha.encrypt(plaintext)
print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")

chacha2 = ChaCha20(key, nonce)
decrypted = chacha2.decrypt(ciphertext)
print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
print(f"\n✅ Uğurlu? {plaintext == decrypted.decode('utf-8')}")


### 6.4 Salsa20 vs ChaCha20 müqayisəsi

In [ ]:
def compare_salsa_chacha():
    """
    Salsa20 və ChaCha20-ni müqayisə edir
    """
    print("\n" + "=" * 80)
    print("📊 SALSA20 vs CHACHA20 MÜQAYİSƏSİ")
    print("=" * 80)

    test_values = [
        (0x11111111, 0x22222222, 0x33333333, 0x44444444),
        (0x01234567, 0x89abcdef, 0x01234567, 0x89abcdef),
        (0x00000000, 0x00000000, 0x00000000, 0x00000000),
        (0xffffffff, 0xffffffff, 0xffffffff, 0xffffffff)
    ]

    print("\n🔍 DÖRDƏBİR RAUND MÜQAYİSƏSİ:")
    print("-" * 80)

    for i, (a, b, c, d) in enumerate(test_values):
        print(f"\nTest {i+1} - Giriş: a={a:08x}, b={b:08x}, c={c:08x}, d={d:08x}")

        s_a, s_b, s_c, s_d = salsa20_quarter_round(a, b, c, d)
        print(f"   Salsa20: {s_a:08x} {s_b:08x} {s_c:08x} {s_d:08x}")

        ch_a, ch_b, ch_c, ch_d = chacha20_quarter_round(a, b, c, d)
        print(f"   ChaCha20: {ch_a:08x} {ch_b:08x} {ch_c:08x} {ch_d:08x}")

        # Fərqlilik dərəcəsi
        diff = (s_a ^ ch_a) | (s_b ^ ch_b) | (s_c ^ ch_c) | (s_d ^ ch_d)
        print(f"   Fərq: {diff:08x}")

    print("\n📌 ÜSTÜNLÜKLƏR:")
    print("   • ChaCha20 daha yaxşı diffuziya təmin edir")
    print("   • ChaCha20-də hər raundda hər söz 2 dəfə dəyişir")
    print("   • ChaCha20-nin raundları asimmetrikdir, bu da təhlükəsizliyi artırır")

compare_salsa_chacha()

### ✍️ Çalışma 2: Salsa20 və ChaCha20 (2 bal)

Aşağıdakı tapşırıqları yerinə yetirin. Bu bölmədə verilmiş `Salsa20` implementasiyasından istifadə edin; onu yenidən sıfırdan yazmaq tələb olunmur.

1. **Salsa20 yoxlanması:** Verilmiş `Salsa20` sinfini test edin və `ChaCha20` sinfi ilə interfeys və davranış baxımından müqayisə edin. Ən azı bir şifrələmə/deşifrələmə testi göstərin.

2. **Sürət müqayisəsi:** Salsa20 və ChaCha20-nin sürətini müqayisə edin (100 MB məlumat şifrləmə).

3. **ARX dizaynı:** ARX dizaynının üstünlükləri nələrdir? Niyə S-qutularından daha sürətlidir?

4. **Diffuziya analizi:** ChaCha20-nin Salsa20-dən əsas fərqləri nələrdir? Niyə daha yaxşı diffuziya təmin edir?

5. **Raund sayları:** Müxtəlif raund sayları üçün (8, 12, 20) təhlükəsizlik və sürət müqayisəsi aparın.
   * 8 raund - sürətli, amma təhlükəsiz deyil
   * 12 raund - Salsa20/12 (eSTREAM versiyası)
   * 20 raund - tam versiya


In [ ]:
# Çalışma 2 - Cavablar

print("📝 ÇALIŞMA 2 CAVABLARI")
print("=" * 80)

print("\n1. SALSA20 İMPLEMENTASİYASI:")
print("   • Yuxarıdakı `Salsa20` sinfi artıq tam və işlək implementasiyadır")
print("   • quarterround, rowround, columnround, doubleround və expand mərhələləri verilib")
print("   • 64-bit nonce və 64-bit blok sayğacı istifadə olunur")
print("   • Rəsmi test vektorları ilə yoxlanılıb")

print("\n2. SÜRƏT MÜQAYİSƏSİ:")
print("   • Salsa20 və ChaCha20 hər ikisi ARX ailəsinə aiddir")
print("   • Praktik sürət platformadan və implementasiyadan asılıdır")
print("   • ChaCha20 çox vaxt proqram təminatında daha rahat istifadə olunur")

print("\n3. ARX ÜSTÜNLÜKLƏRİ:")
print("   • Sadə əməliyyatlar: ADD, XOR, ROT")
print("   • S-qutu yaddaşı tələb etmir")
print("   • Sabit vaxt implementasiyası qurmaq daha asandır")
print("   • Yüngül və kompakt kod yazmaq mümkündür")

print("\n4. DİFFUZİYA FƏRQLƏRİ:")
print("   • Salsa20 və ChaCha20 fərqli quarterround quruluşundan istifadə edir")
print("   • ChaCha20-də sözlər hər cüt raundda fərqli qaydada qarışdırılır")
print("   • Salsa20-də rowround/columnround, ChaCha20-də column/diagonal yanaşması var")

print("\n5. RAUND SAYLARI:")
print("   • Salsa20/8 və Salsa20/12 kimi azaldılmış variantlar mövcuddur")
print("   • Salsa20/20 və ChaCha20 tam variant kimi geniş tanınır")
print("   • TLS 1.3-də istifadə olunan ARX axın şifrəsi ChaCha20-dir, Salsa20 deyil")

## 🔐 ChaCha20-Poly1305 AEAD (25 dəq)

<div style="background-color: #f0f8ff; padding: 15px; border-radius: 10px; border-left: 5px solid #4682b4;">
<h4>🔑 AEAD (Authenticated Encryption with Associated Data)</h4>
<p>AEAD rejimi eyni vaxtda <b>məxfiliyi</b> (şifrləmə) və <b>bütövlüyü</b> (autentifikasiya) təmin edir.</p>
<p>ChaCha20-Poly1305 kombinasiyası:</p>
<ul>
    <li><b>ChaCha20</b> - şifrləmə üçün axın şifrəsi</li>
    <li><b>Poly1305</b> - autentifikasiya üçün universal heş funksiyası</li>
</ul>
<p><b>Üstünlükləri:</b></p>
<ul>
    <li>Sadə və sürətli</li>
    <li>Sabit vaxt implementasiyası mümkün</li>
    <li>TLS 1.3-də standart</li>
</ul>
</div>

### 7.1 Poly1305 MAC

In [ ]:
import secrets


class Poly1305:
    """
    Poly1305 MAC (RFC 8439 ilə uyğun)
    """

    def __init__(self, key):
        if len(key) != 32:
            raise ValueError("❌ Açar 32 bayt olmalıdır")

        r_bytes = key[:16]
        s_bytes = key[16:]

        self.r = int.from_bytes(r_bytes, 'little') & 0x0ffffffc0ffffffc0ffffffc0fffffff
        self.s = int.from_bytes(s_bytes, 'little')
        self.p = (1 << 130) - 5

    @staticmethod
    def _chunks(data, size=16):
        for i in range(0, len(data), size):
            yield data[i:i+size]

    def compute(self, data):
        acc = 0

        for block in self._chunks(data):
            n = int.from_bytes(block + b'\x01', 'little')
            acc = ((acc + n) * self.r) % self.p

        tag = (acc + self.s) % (1 << 128)
        return tag.to_bytes(16, 'little')


print("\n" + "=" * 70)
print("🔏 POLY1305 TESTİ")
print("=" * 70)

# RFC 8439 test vektoru (ictimai, sabit test açarıdır; real istifadə üçün deyil)
rfc_key = bytes.fromhex(
    "85d6be7857556d337f4452fe42d506a8"
    "0103808afb0db2fd4abff6af4149f51b"
)
rfc_msg = b"Cryptographic Forum Research Group"
rfc_expected = bytes.fromhex("a8061dc1305136c6c22b8baf0c0127a9")

poly = Poly1305(rfc_key)
rfc_mac = poly.compute(rfc_msg)
print(f"RFC vektoru uyğun? {rfc_mac == rfc_expected}")
assert rfc_mac == rfc_expected, "Poly1305 RFC 8439 test vektoru uğursuz oldu"

# Sadə demo: açarı sərt kodlaşdırmaq əvəzinə hər işə salmada təsadüfi yaradın
demo_key = secrets.token_bytes(32)
demo_poly = Poly1305(demo_key)
demo_message = b"Salam, bu test mesajidir"
demo_mac = demo_poly.compute(demo_message)
print(f"📨 Mesaj: {demo_message}")
print(f"🔏 MAC: {demo_mac.hex()}")

### 7.2 ChaCha20-Poly1305 AEAD

In [ ]:
import secrets


class ChaCha20Poly1305:
    """
    ChaCha20-Poly1305 AEAD (RFC 8439)

    ⚠️ Təhlükəsizlik xəbərdarlığı: eyni açar ilə hər şifrələmə üçün nonce unikal
    olmalıdır. Heç vaxt eyni key/nonce cütünü təkrar istifadə etməyin. Bu sadə
    tədris implementasiyası hər obyekt üçün yalnız bir şifrələməyə icazə verir.
    """

    def __init__(self, key, nonce):
        if len(key) != 32:
            raise ValueError("❌ Açar 32 bayt olmalıdır")
        if len(nonce) != 12:
            raise ValueError("❌ Nonce 12 bayt olmalıdır")

        self.key = key
        self.nonce = nonce
        self._used_for_encryption = False

    @staticmethod
    def _pad16(data):
        remainder = len(data) % 16
        if remainder == 0:
            return b''
        return b'\x00' * (16 - remainder)

    def _poly1305_key(self):
        chacha_zero = ChaCha20(self.key, self.nonce, 0)
        return chacha_zero.generate_keystream(64)[:32]

    def _build_mac_data(self, aad, ciphertext):
        return (
            aad +
            self._pad16(aad) +
            ciphertext +
            self._pad16(ciphertext) +
            len(aad).to_bytes(8, 'little') +
            len(ciphertext).to_bytes(8, 'little')
        )

    def encrypt_and_auth(self, plaintext, aad=b''):
        if self._used_for_encryption:
            raise ValueError("❌ Bu obyekt artıq şifrələmə üçün istifadə olunub; eyni key/nonce cütünü təkrar istifadə etməyin")

        if isinstance(plaintext, str):
            plaintext = plaintext.encode('utf-8')
        if isinstance(aad, str):
            aad = aad.encode('utf-8')

        poly = Poly1305(self._poly1305_key())

        chacha = ChaCha20(self.key, self.nonce, 1)
        ciphertext = chacha.encrypt(plaintext)

        mac_data = self._build_mac_data(aad, ciphertext)
        tag = poly.compute(mac_data)
        self._used_for_encryption = True

        return ciphertext, tag

    def decrypt_and_verify(self, ciphertext, tag, aad=b''):
        if isinstance(aad, str):
            aad = aad.encode('utf-8')

        poly = Poly1305(self._poly1305_key())
        expected_tag = poly.compute(self._build_mac_data(aad, ciphertext))

        if not self._constant_time_compare(tag, expected_tag):
            raise ValueError("❌ Autentifikasiya uğursuz oldu!")

        chacha = ChaCha20(self.key, self.nonce, 1)
        return chacha.decrypt(ciphertext)

    @staticmethod
    def _constant_time_compare(a, b):
        if len(a) != len(b):
            return False
        result = 0
        for x, y in zip(a, b):
            result |= x ^ y
        return result == 0


print("\n" + "=" * 80)
print("🔐 CHACHA20-POLY1305 AEAD TESTİ")
print("=" * 80)
print("⚠️  Qayda: sabit açar üçün hər şifrələmədə unikal nonce istifadə edin; key/nonce cütünü heç vaxt təkrarlamayın.")

# RFC 8439 test vektoru: bu sabit dəyərlər yalnız uyğunluğu yoxlamaq üçündür.
rfc_key = bytes.fromhex(
    "1c9240a5eb55d38af333888604f6b5f0"
    "473917c1402b80099dca5cbc207075c0"
)
rfc_nonce = bytes.fromhex("000000000102030405060708")
rfc_aad = bytes.fromhex("f33388860000000000004e91")
rfc_plaintext = bytes.fromhex(
    "496e7465726e65742d4472616674732061726520647261667420646f63756d656e74732076616c696420"
    "666f722061206d6178696d756d206f6620736978206d6f6e74687320616e64206d617920626520757064"
    "617465642c207265706c616365642c206f72206f62736f6c65746564206279206f7468657220646f6375"
    "6d656e747320617420616e792074696d652e20497420697320696e617070726f70726961746520746f20"
    "75736520496e7465726e65742d447261667473206173207265666572656e6365206d6174657269616c20"
    "6f7220746f2063697465207468656d206f74686572207468616e206173202fe2809c776f726b20696e20"
    "70726f67726573732e2fe2809d"
)
rfc_expected_ciphertext = bytes.fromhex(
    "64a0861575861af460f062c79be643bd"
    "5e805cfd345cf389f108670ac76c8cb2"
    "4c6cfc18755d43eea09ee94e382d26b0"
    "bdb7b73c321b0100d4f03b7f355894cf"
    "332f830e710b97ce98c8a84abd0b9481"
    "14ad176e008d33bd60f982b1ff37c855"
    "9797a06ef4f0ef61c186324e2b350638"
    "3606907b6a7c02b0f9f6157b53c867e4"
    "b9166c767b804d46a59b5216cde7a4e9"
    "9040c5a40433225ee282a1b0a06c523e"
    "af4534d7f83fa1155b0047718cbc546a"
    "0d072b04b3564eea1b422273f548271a"
    "0bb2316053fa76991955ebd63159434e"
    "cebb4e466dae5a1073a6727627097a10"
    "49e617d91d361094fa68f0ff77987130"
    "305beaba2eda04df997b714d6c6f2c29"
    "a6ad5cb4022b02709b"
)
rfc_expected_tag = bytes.fromhex("eead9d67890cbb22392336fea1851f38")

aead = ChaCha20Poly1305(rfc_key, rfc_nonce)
ciphertext, tag = aead.encrypt_and_auth(rfc_plaintext, rfc_aad)

print(f"RFC ciphertext uyğun? {ciphertext == rfc_expected_ciphertext}")
print(f"RFC tag uyğun? {tag == rfc_expected_tag}")
assert ciphertext == rfc_expected_ciphertext, "ChaCha20-Poly1305 ciphertext RFC vektoruna uyğun deyil"
assert tag == rfc_expected_tag, "ChaCha20-Poly1305 tag RFC vektoruna uyğun deyil"

# Sadə demo: real istifadədə açarı təhlükəsiz saxlayın və hər mesaj üçün yeni nonce yaradın.
demo_key = secrets.token_bytes(32)
demo_nonce = secrets.token_bytes(12)
demo_aead = ChaCha20Poly1305(demo_key, demo_nonce)

plaintext = "Bu çox gizli mesajdır!"
aad = "header məlumatı"

print(f"📨 Orijinal mətn: {plaintext}")
print(f"📎 AAD: {aad}")
print(f"🎲 Bu mesaj üçün nonce (hex): {demo_nonce.hex()}")

ciphertext, tag = demo_aead.encrypt_and_auth(plaintext, aad)
print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")
print(f"🔏 MAC tag (hex): {tag.hex()}")

try:
    decrypted = demo_aead.decrypt_and_verify(ciphertext, tag, aad)
    print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
    print("✅ Autentifikasiya uğurlu!")
except ValueError as e:
    print(f"❌ {e}")

print("\n🔍 SƏHV TAG TESTİ:")
wrong_tag = bytearray(tag)
wrong_tag[0] ^= 1

try:
    demo_aead.decrypt_and_verify(ciphertext, bytes(wrong_tag), aad)
    print("❌ Bu işləməməlidir!")
except ValueError as e:
    print(f"✅ Düzgün: {e}")

### ✍️ Çalışma 3: ChaCha20-Poly1305 (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Fayl şifrləmə:** ChaCha20-Poly1305 AEAD-dən istifadə edərək fayl şifrləyən proqram yazın.

2. **MAC səhvləri:** MAC dəyərini dəyişdirin və autentifikasiyanın necə işlədiyini yoxlayın.

3. **AAD-nin rolu:** Niyə AAD (Associated Data) vacibdir? Şifrlənməyən, ancaq autentifikasiya edilən məlumatlara nümunələr verin.

4. **TLS 1.3:** TLS 1.3 protokolunda ChaCha20-Poly1305-in rolu nədir? Hansı vəziyyətlərdə AES-GCM-ə üstün tutulur?

5. **Google-un seçimi:** Google nə üçün mobil cihazlarda ChaCha20-ni AES-ə üstün tutur?

In [ ]:
# Çalışma 3 - Cavablar

print("📝 ÇALIŞMA 3 CAVABLARI")
print("=" * 80)

print("\n1. FAYL ŞİFRLƏMƏ:")
print("""
def encrypt_file(filename, key, nonce, aad=b''):
    aead = ChaCha20Poly1305(key, nonce)
    with open(filename, 'rb') as f:
        data = f.read()
    ciphertext, tag = aead.encrypt_and_auth(data, aad)
    with open(filename + '.enc', 'wb') as f:
        f.write(ciphertext)
    with open(filename + '.tag', 'wb') as f:
        f.write(tag)
""")

print("\n2. MAC SƏHVLƏRİ:")
print("   • Tag 1 bit dəyişsə belə yoxlama uğursuz olur")
print("   • Bu, məlumat bütövlüyünü və dəyişdirilməməsini təmin edir")
print("   • Hücumçu ciphertext-i səssizcə dəyişdirə bilməz")

print("\n3. AAD-NİN ROLU:")
print("   • AAD şifrlənmir, amma autentifikasiya edilir")
print("   • Nümunələr: başlıqlar, paket nömrələri, protokol parametrləri")
print("   • Beləliklə metadata da dəyişiklikdən qorunur")

print("\n4. TLS 1.3:")
print("   • ChaCha20-Poly1305 TLS 1.3-də istifadə olunan əsas AEAD rejimlərindən biridir")
print("   • AES sürətləndirilməsi zəif olan cihazlarda tez-tez üstün seçim olur")
print("   • Həm AES-GCM, həm də ChaCha20-Poly1305 praktikada geniş yayılıb")

print("\n5. GOOGLE-UN SEÇİMİ:")
print("   • Proqram təminatında yüksək performans göstərir")
print("   • Mobil və müxtəlif CPU-larda sabit davranış verir")
print("   • Sabit vaxtlı implementasiya qurmaq asandır")

## 📚 eSTREAM portfelinin digər şifrələri (20 dəq)

### 8.1 Grain v1 şifrəsi

<div style="background-color: #fff3e0; padding: 15px; border-radius: 10px; border-left: 5px solid #ff9800;">
<h4>📜 Grain v1</h4>
<p>Bu notebook-da artıq sadələşdirilmiş demo deyil, eSTREAM Profil 2 portfelinə daxil olan <b>Grain v1</b> quruluşu göstərilir.</p>
<ul>
    <li><b>Açar:</b> 80-bit</li>
    <li><b>IV:</b> 64-bit</li>
    <li><b>Daxili vəziyyət:</b> 160-bit (LFSR 80-bit + NFSR 80-bit)</li>
    <li><b>İlkinləşdirmə:</b> 160 dövr</li>
</ul>
</div>

In [ ]:
class Grain:
    """
    Grain v1 axın şifrəsi.

    Qeyd:
    - Bu sinif eSTREAM portfelindəki Grain v1 strukturunu izləyir.
    - Bayt -> bit çevrilməsində LSB-first xəritələnməsi istifadə olunur
      (istinad implementasiyalarına uyğun rahat byte wrapper).
    """

    def __init__(self, key, iv):
        if len(key) != 10 or len(iv) != 8:
            raise ValueError("❌ Grain v1 üçün key 10 bayt, IV 8 bayt olmalıdır")

        self.key = key
        self.iv = iv
        self._init_state()

    @staticmethod
    def _bytes_to_bits_lsb(data, total_bits=None):
        bits = []
        for byte in data:
            for i in range(8):
                bits.append((byte >> i) & 1)
        if total_bits is not None:
            bits = bits[:total_bits]
        return bits

    @staticmethod
    def _bits_to_bytes_lsb(bits):
        out = bytearray((len(bits) + 7) // 8)
        for i, bit in enumerate(bits):
            if bit:
                out[i // 8] |= 1 << (i % 8)
        return bytes(out)

    def _init_state(self):
        # Grain v1: NFSR <- key, LFSR <- IV || 1^16
        self.nfsr = self._bytes_to_bits_lsb(self.key, 80)
        self.lfsr = self._bytes_to_bits_lsb(self.iv, 64) + [1] * 16

        # 160 ilkinləşdirmə dövrü: çıxış biti hər iki registrə geri qidalanır
        for _ in range(160):
            z = self._output()
            self._clock(initializing=True, feedback_bit=z)

    def _h(self):
        x0 = self.lfsr[3]
        x1 = self.lfsr[25]
        x2 = self.lfsr[46]
        x3 = self.lfsr[64]
        x4 = self.nfsr[63]

        return (
            x1 ^ x4 ^
            (x0 & x3) ^ (x2 & x3) ^ (x3 & x4) ^
            (x0 & x1 & x2) ^ (x0 & x2 & x3) ^
            (x0 & x2 & x4) ^ (x1 & x2 & x4) ^
            (x2 & x3 & x4)
        )

    def _output(self):
        b = self.nfsr
        return b[1] ^ b[2] ^ b[4] ^ b[10] ^ b[31] ^ b[43] ^ b[56] ^ self._h()

    def _lfsr_feedback(self):
        s = self.lfsr
        return s[0] ^ s[13] ^ s[23] ^ s[38] ^ s[51] ^ s[62]

    def _nfsr_feedback(self):
        b = self.nfsr
        s = self.lfsr

        return (
            s[0] ^
            b[0] ^ b[9] ^ b[14] ^ b[21] ^ b[28] ^ b[33] ^
            b[37] ^ b[45] ^ b[52] ^ b[60] ^ b[62] ^
            (b[63] & b[60]) ^ (b[37] & b[33]) ^ (b[15] & b[9]) ^
            (b[60] & b[52] & b[45]) ^
            (b[33] & b[28] & b[21]) ^
            (b[63] & b[45] & b[28] & b[9]) ^
            (b[60] & b[52] & b[37] & b[33]) ^
            (b[63] & b[60] & b[21] & b[15]) ^
            (b[63] & b[60] & b[52] & b[45] & b[37]) ^
            (b[33] & b[28] & b[21] & b[15] & b[9]) ^
            (b[52] & b[45] & b[37] & b[33] & b[28] & b[21])
        )

    def _clock(self, initializing=False, feedback_bit=0):
        new_l = self._lfsr_feedback()
        new_n = self._nfsr_feedback()

        if initializing:
            new_l ^= feedback_bit
            new_n ^= feedback_bit

        self.lfsr = self.lfsr[1:] + [new_l]
        self.nfsr = self.nfsr[1:] + [new_n]

    def next_bit(self):
        z = self._output()
        self._clock(initializing=False)
        return z

    def generate_keystream_bits(self, n):
        return [self.next_bit() for _ in range(n)]

    def generate_keystream(self, nbytes):
        bits = self.generate_keystream_bits(nbytes * 8)
        return self._bits_to_bytes_lsb(bits)

    def encrypt(self, plaintext):
        if isinstance(plaintext, str):
            plaintext = plaintext.encode('utf-8')

        keystream = self.generate_keystream(len(plaintext))
        return bytes(p ^ k for p, k in zip(plaintext, keystream))

    def decrypt(self, ciphertext):
        return self.encrypt(ciphertext)


# Test
print("\n" + "=" * 70)
print("🌾 GRAIN v1 TESTİ")
print("=" * 70)

key = bytes(range(10))
iv = bytes(range(8))
grain = Grain(key, iv)

keystream16 = grain.generate_keystream(16)
print(f"İlk 16 bayt açar axını: {keystream16.hex()}")

plaintext = "Grain v1 artıq sadələşdirilmiş deyil."
ciphertext = Grain(key, iv).encrypt(plaintext)
decrypted = Grain(key, iv).decrypt(ciphertext)

print(f"📨 Orijinal mətn: {plaintext}")
print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")
print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")
print(f"✅ Uğurlu? {plaintext == decrypted.decode('utf-8')}")

### 8.2 Digər mühüm şifrələr

**SNOW 3G** - 3GPP mobil rabitə standartında istifadə olunan axın şifrəsidir. 128-bit açar və 128-bit IV istifadə edir. LFSR və Sonlu Vəziyyət Maşınından (FSM) ibarətdir.

**ZUC** - Çin milli standartı olan, 4G LTE şəbəkələrində istifadə edilən axın şifrəsidir. ZUC-128 (128-bit açar) və ZUC-256 (256-bit açar) versiyaları mövcuddur.

### 8.3 Müqayisə cədvəli

| **Şifrə** | **Açar (bit)** | **IV (bit)** | **Daxili vəziyyət** | **Tətbiq sahəsi** |
|-----------|----------------|--------------|---------------------|-------------------|
| Trivium | 80 | 80 | 288 bit | RFID, IoT |
| Grain v1 | 80 | 64 | 160 bit | Məhdud aparat |
| ChaCha20 | 256 | 96 | 512 bit | TLS, VPN, HTTPS |
| Salsa20 | 256 | 64 | 512 bit | Proqram təminatı |
| SNOW 3G | 128 | 128 | 576 bit | 3G/4G mobil |
| ZUC | 128/256 | 128/184 | 560 bit | 4G LTE |
| HC-128 | 128 | 128 | 1024 bit | Proqram təminatı |

### ✍️ Çalışma 4: eSTREAM portfeli (1.5 bal)

Aşağıdakı tapşırıqları yerinə yetirin:

1. **Portfel üzvləri:** eSTREAM portfelinə daxil olan Profil 1 və Profil 2 şifrələrini sadalayın.

2. **Xüsusiyyətlər:** Hər bir şifrə üçün əsas xüsusiyyətləri qeyd edin (açar uzunluğu, IV uzunluğu, daxili vəziyyət).

3. **Grain analizi:** Grain şifrəsinin LFSR və NLFSR quruluşunu izah edin. Niyə həm LFSR, həm də NLFSR istifadə olunur?

4. **Mobil standartlar:** SNOW 3G və ZUC şifrələri hansı standartlarda istifadə olunur? Onların təhlükəsizlik səviyyəsi nədir?

5. **Yüngüllük:** Niyə Trivium ən yüngül şifrələrdən biri hesab olunur? Hansı tətbiqlər üçün idealdır?

In [ ]:
# Çalışma 4 - Cavablar

print("📝 ÇALIŞMA 4 CAVABLARI")
print("=" * 80)

print("\n1. eSTREAM PORTFELİ:")
print("   Profil 1 (proqram): HC-128, Rabbit, Salsa20/12, SOSEMANUK")
print("   Profil 2 (aparat): Grain v1, MICKEY 2.0, Trivium")

print("\n2. XÜSUSİYYƏTLƏR:")
print("   • HC-128: 128-bit açar, 128-bit IV, 1024-bit vəziyyət")
print("   • Rabbit: 128-bit açar, 64-bit IV, 513-bit daxili vəziyyət")
print("   • Salsa20: 256-bit açar, 64-bit nonce, 512-bit daxili vəziyyət")
print("   • Grain v1: 80-bit açar, 64-bit IV, 160-bit vəziyyət")
print("   • Trivium: 80-bit açar, 80-bit IV, 288-bit vəziyyət")

print("\n3. GRAIN v1 ANALİZİ:")
print("   • LFSR xətti komponentdir və uzun period verməyə kömək edir")
print("   • NFSR qeyri-xətti komponentdir və təhlükəsizliyi artırır")
print("   • Çıxış funksiyası hər iki registrdən bitlər götürərək qarışdırma yaradır")

print("\n4. MOBİL STANDARTLAR:")
print("   • SNOW 3G: 3GPP UEA2/UIA2")
print("   • ZUC: 3GPP EEA3/EIA3")
print("   • Hər ikisi mobil rabitə standartlarında istifadə olunur")

print("\n5. YÜNGÜLLÜK:")
print("   • Trivium və Grain v1 məhdud aparat mühitləri üçün nəzərdə tutulub")
print("   • Kiçik registr quruluşları və sadə bit əməliyyatları istifadə edirlər")
print("   • RFID, sensor və IoT ssenarilərində faydalıdırlar")

## 🖥️ İnteqrasiya edilmiş tətbiq (15 dəq)

In [ ]:
def modern_stream_menu():
    """
    Müasir axın şifrələri laboratoriyası - interaktiv menyu
    """
    def read_exact_bytes(prompt, expected_len):
        """İstifadəçidən dəqiq `expected_len` bayt uzunluğunda giriş al."""
        while True:
            data = input(prompt).encode()
            if len(data) == expected_len:
                return data
            print(f"❌ Giriş dəqiq {expected_len} bayt olmalıdır; daxil etdiniz: {len(data)} bayt. Yenidən cəhd edin.")

    while True:
        print("\n" + "=" * 70)
        print("🔐 MÜASİR AXIN ŞİFRƏLƏRİ LABORATORİYASI - Məşğələ 7")
        print("=" * 70)
        print("1. 🔧 Trivium şifrəsi")
        print("2. 🔄 Salsa20 dörddəbir raund")
        print("3. 🔄 ChaCha20 dörddəbir raund")
        print("4. 🔐 ChaCha20 axın şifrəsi")
        print("5. 🔏 ChaCha20-Poly1305 AEAD")
        print("6. 📊 Müqayisə: Salsa20 vs ChaCha20")
        print("7. 📚 eSTREAM portfeli haqqında məlumat")
        print("8. 📈 Trivium statistik analizi")
        print("0. ❌ Çıxış")
        print("=" * 70)

        choice = input("📌 Seçiminiz: ")

        if choice == '1':
            print("\n--- TRIVIUM ---")
            key = read_exact_bytes("Açar (10 bayt, məsələn, 0123456789): ", 10)
            iv = read_exact_bytes("IV (10 bayt, məsələn, abcdefghij): ", 10)
            text = input("Şifrələnəcək mətn: ")

            trivium = Trivium(key, iv)
            ciphertext = trivium.encrypt(text)
            print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")

            trivium.reset()
            decrypted = trivium.decrypt(ciphertext)
            print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")

        elif choice == '2':
            print("\n--- SALSA20 DÖRDƏBİR RAUND ---")
            a = int(input("a (hex, məsələn, 11111111): "), 16)
            b = int(input("b (hex): "), 16)
            c = int(input("c (hex): "), 16)
            d = int(input("d (hex): "), 16)

            a, b, c, d = salsa20_quarter_round(a, b, c, d)
            print(f"📤 Nəticə: {a:08x} {b:08x} {c:08x} {d:08x}")

        elif choice == '3':
            print("\n--- CHACHA20 DÖRDƏBİR RAUND ---")
            a = int(input("a (hex, məsələn, 11111111): "), 16)
            b = int(input("b (hex): "), 16)
            c = int(input("c (hex): "), 16)
            d = int(input("d (hex): "), 16)

            a, b, c, d = chacha20_quarter_round(a, b, c, d)
            print(f"📤 Nəticə: {a:08x} {b:08x} {c:08x} {d:08x}")

        elif choice == '4':
            print("\n--- CHACHA20 AXIN ŞİFRƏSİ ---")
            key = read_exact_bytes("Açar (32 bayt, məsələn, 32 dənə 0-9): ", 32)
            nonce = read_exact_bytes("Nonce (12 bayt): ", 12)
            text = input("Şifrələnəcək mətn: ")

            chacha = ChaCha20(key, nonce)
            ciphertext = chacha.encrypt(text)
            print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")

            chacha2 = ChaCha20(key, nonce)
            decrypted = chacha2.decrypt(ciphertext)
            print(f"🔓 Deşifrələnmiş: {decrypted.decode('utf-8')}")

        elif choice == '5':
            print("\n--- CHACHA20-POLY1305 AEAD ---")
            key = read_exact_bytes("Açar (32 bayt): ", 32)
            nonce = read_exact_bytes("Nonce (12 bayt): ", 12)
            text = input("Şifrələnəcək mətn: ")
            aad = input("Əlaqəli məlumat (AAD, boş ola bilər): ").encode()

            aead = ChaCha20Poly1305(key, nonce)
            ciphertext, tag = aead.encrypt_and_auth(text.encode(), aad)
            print(f"🔒 Şifrəli mətn (hex): {ciphertext.hex()}")
            print(f"🔏 MAC tag (hex): {tag.hex()}")

            try:
                decrypted = aead.decrypt_and_verify(ciphertext, tag, aad)
                print(f"🔓 Deşifrələnmiş: {decrypted.decode()}")
                print("✅ Autentifikasiya uğurlu!")
            except ValueError as e:
                print(f"❌ {e}")

        elif choice == '6':
            compare_salsa_chacha()

        elif choice == '7':
            print("\n--- eSTREAM PORTFELİ ---")
            print("\n📌 Profil 1 (proqram):")
            print("   • HC-128 - 128-bit açar, yüksək sürət")
            print("   • Rabbit - 128-bit açar, patent pulsuz")
            print("   • Salsa20/12 - 256-bit açar, ARX dizaynı")
            print("   • SOSEMANUK - 128-bit açar, SNOW + Serpent")
            print("\n📌 Profil 2 (aparat):")
            print("   • Grain v1 - 80-bit açar, LFSR + NFSR")
            print("   • MICKEY - 80-bit açar, qeyri-müntəzəm saatlama")
            print("   • Trivium - 80-bit açar, ən yüngül")

        elif choice == '8':
            analyze_trivium()

        elif choice == '0':
            print("👋 Proqram bitdi. Sağ olun!")
            break

        else:
            print("❌ Yanlış seçim. Yenidən cəhd edin.")

# Proqramı işə sal (istəyə bağlı)
# modern_stream_menu()


## 🏠 Ev tapşırığı

### 📦 Ev tapşırığı 1: ChaCha20 paketi (3 bal)

Aşağıdakı funksiyaları özündə birləşdirən Python paketi yazın:

```
chacha_package/
├── __init__.py
├── chacha20.py          # ChaCha20 sinfi, blok funksiyası, şifrləmə/deşifrləmə
├── poly1305.py          # Poly1305 MAC sinfi
├── chacha20poly1305.py  # AEAD rejimi
├── salsa20.py           # Salsa20 sinfi (əlavə)
├── trivium.py           # Trivium sinfi
└── main.py              # İnteraktiv menyu
```

**Tələblər:**
* Hər bir funksiya üçün docstring yazın
* Səhv hallarını idarə edin (try-except)
* Kod təmiz və oxunaqlı olmalıdır
* Hər modul üçün test nümunələri əlavə edin

### 🔐 Ev tapşırığı 2: Praktik məsələlər (2 bal)

Aşağıdakı məsələləri həll edin:

1. **Sürət testi:** ChaCha20 istifadə edərək 1 GB faylı şifrləyin və sürətini ölçün. Nəticəni AES ilə müqayisə edin (cryptography kitabxanasından istifadə edə bilərsiniz).

2. **Açar təkrarı:** Eyni açar və nonce ilə iki fərqli mesaj şifrləyin. Nə baş verir? Bu, OTP-dəki açar təkrarı problemindən nə ilə fərqlənir?

3. **MAC sınağı:** ChaCha20-Poly1305 ilə şifrlənmiş mesajın MAC dəyərini dəyişdirin və nə baş verdiyini yoxlayın. Bu, hansı təhlükəsizlik xidmətini təmin edir?

4. **Müqayisə:** Trivium və ChaCha20-nin sürətini müqayisə edin. Hansı daha sürətlidir? Niyə?

5. **ARX dizaynı:** Öz ARX əsaslı axın şifrənizi dizayn etməyə çalışın. Onun təhlükəsizliyini necə test edərdiniz?

### 📚 Ev tapşırığı 3: Tədqiqat (2 bal)

Araşdırma aparın və aşağıdakı suallara cavab tapın. Cavablarınızı 1-2 səhifəlik hesabat şəklində təqdim edin:

1. **TLS 1.3:** TLS 1.3-də niyə məhz ChaCha20-Poly1305 və AES-GCM dəstəklənir? Hansı vəziyyətlərdə hansı üstünlük təşkil edir?

2. **Hardware sürətləndirmə:** AES-in hardware sürətləndirilməsi olmayan cihazlarda (mobil, IoT) niyə ChaCha20 üstünlük təşkil edir? Rəqəmsal məlumatlar (benchmarks) göstərin.

3. **Mobil şəbəkələr:** SNOW 3G və ZUC şifrələrinin təhlükəsizlik analizi haqqında nə məlumat var? Onların kriptoanalizi aparılıbmı?

4. **eSTREAM-dən sonra:** eSTREAM layihəsindən sonra hansı yeni axın şifrələri təklif edilib? Onlar hansı üstünlüklərə malikdir?

5. **Kvant təhlükəsi:** Kvant kompüterlər müasir axın şifrələrinə təhlükə yaradırmı? Hansı şifrələr kvant təhlükəsinə qarşı daha davamlıdır?

**Format tələbləri:**
* PDF formatında təqdim edin
* Ən azı 5 qaynaq göstərin (kitab, məqalə, veb səhifə)
* Öz sözlərinizlə yazın (copy-paste yox)
* Mümkünsə, qrafiklər və ya performans müqayisələri əlavə edin

## 📌 Yekun və müzakirə sualları

<div style="background-color: #e8f4f8; padding: 15px; border-radius: 10px; border-left: 5px solid #2c3e50;">
<h3>📋 Xülasə</h3>
<p>Bu məşğələdə öyrəndiklərimiz:</p>
<ul>
    <li>✅ <b>eSTREAM layihəsi</b> - müasir axın şifrələrinin seçilməsi və standartlaşdırılması</li>
    <li>✅ <b>Trivium</b> - ən yüngül axın şifrəsi, 288-bit daxili vəziyyət, 80-bit təhlükəsizlik</li>
    <li>✅ <b>ARX dizaynı</b> - Add, Rotate, XOR əməliyyatları ilə qeyri-xəttilik</li>
    <li>✅ <b>Salsa20</b> - eSTREAM Profil 1 qalibi, 256-bit açar, 20 raund</li>
    <li>✅ <b>ChaCha20</b> - Salsa20-nin təkmilləşdirilmiş versiyası, daha yaxşı diffuziya</li>
    <li>✅ <b>ChaCha20-Poly1305</b> - AEAD rejimi, TLS 1.3-də standart</li>
    <li>✅ <b>Grain, SNOW 3G, ZUC</b> - xüsusi tətbiq sahələri üçün şifrələr</li>
</ul>
<p><i>Müasir axın şifrələri müxtəlif dizayn yanaşmalarından istifadə edir: registr-əsaslı (Trivium, Grain v1) və ARX-əsaslı (Salsa20, ChaCha20). Hər yanaşmanın üstünlükləri tətbiq sahəsindən və platformadan asılıdır.</i></p>
</div>

### 💭 Müzakirə sualları

1. Bu məşğələdə ən maraqlı tapdığınız nə oldu?
2. ChaCha20-nin Salsa20-dən üstünlükləri nələrdir?
3. Niyə müasir protokollarda AEAD rejimi vacibdir?
4. Trivium kimi 80-bit şifrələr bu gün hələ də təhlükəsiz hesab edilirmi?
5. AES hardware sürətləndirilməsi olmayan cihazlarda niyə ChaCha20 seçilir?
6. TLS 1.3-də niyə həm ChaCha20, həm də AES dəstəklənir?
7. Gələcəkdə axın şifrələri hansı istiqamətdə inkişaf edəcək?

## 📚 Əlavə resurslar

* 📘 **eSTREAM layihəsi:** [https://www.ecrypt.eu.org/stream/](https://www.ecrypt.eu.org/stream/)
* 📙 **ChaCha20 və Poly1305 RFC:** [https://tools.ietf.org/html/rfc8439](https://tools.ietf.org/html/rfc8439)
* 📗 **Daniel Bernstein-in səhifəsi:** [https://cr.yp.to/chacha.html](https://cr.yp.to/chacha.html)
* 📕 **Trivium spesifikasiyası:** [https://www.ecrypt.eu.org/stream/p3ciphers/trivium/trivium_p3.pdf](https://www.ecrypt.eu.org/stream/p3ciphers/trivium/trivium_p3.pdf)
* 📘 **SNOW 3G spesifikasiyası:** [https://www.gsma.com/aboutus/wp-content/uploads/2014/12/uea2_uia2_specification_v1_01.pdf](https://www.gsma.com/aboutus/wp-content/uploads/2014/12/uea2_uia2_specification_v1_01.pdf)
* 📙 **ZUC spesifikasiyası:** [http://www.is.cas.cn/ztzl2016/zuc/](http://www.is.cas.cn/ztzl2016/zuc/)
* 📗 **TLS 1.3 RFC:** [https://tools.ietf.org/html/rfc8446](https://tools.ietf.org/html/rfc8446)
* 📘 **Google-nın ChaCha20 təcrübəsi:** [https://www.imperialviolet.org/2013/12/26/chacha20.html](https://www.imperialviolet.org/2013/12/26/chacha20.html)

---

<div style="background-color: #f0f0f0; padding: 20px; border-radius: 10px; text-align: center;">
<h2>✅ Məşğələ 7 tamamlandı!</h2>
<p>Bütün kodları və tapşırıq cavablarını növbəti məşğələyə qədər təqdim edin.</p>
<p><em>Kodlar aydın şərhlərlə yazılmalı və asan oxunmalıdır. Hər bir funksiyanın nə etdiyini izah edən şərhlər əlavə edin.</em></p>
<p style="font-size: 1.2em; margin-top: 15px;">🔐 <b>Müasir axın şifrələri - gündəlik həyatımızın gizli qəhrəmanları!</b></p>
</div>